# Jane Street Market Forecasting - LGBM Baseline (Offline, Memory-Aware)

This notebook builds a simple offline LightGBM baseline for `responder_6` with a focus on running on limited RAM:

- **Polars-first** pipeline (lazy scan + feature engineering in Polars)
- Aggressive but safe dtype downcasting (`float32`, compact ints)
- Convert to pandas **only per fold** right before training
- Time-series-aware 3-fold CV with each validation fold close to **6 months** of `date_id`
- Competition metric: **sample-weighted zero-mean R^2**

This is intentionally for local offline iteration only.

In [1]:
# Uncomment and run once if needed.
# !pip install lightgbm polars pyarrow pandas scikit-learn

import gc
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [2]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "jane-street-market-forecasting":
    PROJECT_DIR = PROJECT_DIR / "jane-street-market-forecasting"

KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/jane-street-real-time-market-data-forecasting")
IS_KAGGLE = KAGGLE_DATA_DIR.exists()

DATA_DIR = KAGGLE_DATA_DIR if IS_KAGGLE else (PROJECT_DIR / "data")
TRAIN_DIR = DATA_DIR / "train.parquet"
TRAIN_GLOB = str(TRAIN_DIR / "partition_id=*" / "*.parquet")

TARGET = "responder_6"
WEIGHT_COL = "weight"
FEATURE_COLS = [f"feature_{i:02d}" for i in range(79)]
RESPONDER_COLS = [f"responder_{i}" for i in range(9)]
BASE_COLS = ["date_id", "time_id", "symbol_id", WEIGHT_COL] + FEATURE_COLS + RESPONDER_COLS

N_FOLDS = 1
DATE_GAP = 1
VAL_DAYS = 120  # roughly 6 months by date_id
TRAIN_LOOKBACK_DAYS = 360  # cap train window for memory; set None for full expanding window

# Optional cap for quick experiments / memory safety.
# Keep >= (N_FOLDS * VAL_DAYS + 180) for meaningful folds.
MAX_DATES = 500

# Phase 1 feature engineering: cross-sectional features on a compact selected subset.
PHASE1_ENABLE = False
PHASE1_TOPK_BASE_FEATURES = 10

# Phase 2 feature engineering: per-symbol lag features on selected base features.
PHASE2_ENABLE = True
PHASE2_TOPK_BASE_FEATURES = 8
PHASE2_LAGS = [1, 2]

# Phase 3 feature engineering: previous-day lag features.
PHASE3_ENABLE = True
PHASE3_USE_RESPONDER_LAGS = True

print(f"Running on Kaggle: {IS_KAGGLE}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"DATA_DIR exists: {DATA_DIR.exists()}")
print(f"TRAIN_DIR exists: {TRAIN_DIR.exists()}")
print(f"TRAIN_LOOKBACK_DAYS: {TRAIN_LOOKBACK_DAYS}")
print(f"MAX_DATES: {MAX_DATES}")
print(f"PHASE1_TOPK_BASE_FEATURES: {PHASE1_TOPK_BASE_FEATURES}")
print(f"PHASE2_ENABLE: {PHASE2_ENABLE}")
print(f"PHASE2_TOPK_BASE_FEATURES: {PHASE2_TOPK_BASE_FEATURES}")
print(f"PHASE2_LAGS: {PHASE2_LAGS}")
print(f"PHASE3_ENABLE: {PHASE3_ENABLE}")
print(f"PHASE3_USE_RESPONDER_LAGS: {PHASE3_USE_RESPONDER_LAGS}")

DATA_DIR exists: True
TRAIN_DIR exists: True


In [3]:
def build_scan() -> pl.LazyFrame:
    scan = pl.scan_parquet(TRAIN_GLOB).select(BASE_COLS)

    if MAX_DATES is not None:
        max_date = scan.select(pl.max("date_id").alias("max_date")).collect().item()
        min_date = max_date - MAX_DATES + 1
        scan = scan.filter(pl.col("date_id") >= min_date)

    # Downcast to reduce memory pressure while preserving practical precision.
    cast_exprs = [
        pl.col("date_id").cast(pl.Int16),
        pl.col("time_id").cast(pl.Int16),
        pl.col("symbol_id").cast(pl.Int16),
        pl.col(WEIGHT_COL).cast(pl.Float32),
    ] + [pl.col(c).cast(pl.Float32) for c in FEATURE_COLS + RESPONDER_COLS]

    scan = scan.with_columns(cast_exprs)
    return scan


scan = build_scan()
df_pl = scan.collect(streaming=True).sort(["date_id", "time_id", "symbol_id"])

print(df_pl.shape)
print(df_pl.select(["date_id", "time_id", "symbol_id", WEIGHT_COL, TARGET]).head())
print(f"Estimated in-memory size (MB): {df_pl.estimated_size('mb'):.2f}")

(26659688, 84)
shape: (5, 5)
┌─────────┬─────────┬───────────┬──────────┬─────────────┐
│ date_id ┆ time_id ┆ symbol_id ┆ weight   ┆ responder_6 │
│ ---     ┆ ---     ┆ ---       ┆ ---      ┆ ---         │
│ i16     ┆ i16     ┆ i16       ┆ f32      ┆ f32         │
╞═════════╪═════════╪═══════════╪══════════╪═════════════╡
│ 969     ┆ 0       ┆ 0         ┆ 3.269281 ┆ 1.121171    │
│ 969     ┆ 0       ┆ 1         ┆ 6.936615 ┆ 1.147567    │
│ 969     ┆ 0       ┆ 2         ┆ 2.6898   ┆ 0.023757    │
│ 969     ┆ 0       ┆ 3         ┆ 2.604316 ┆ -0.091458   │
│ 969     ┆ 0       ┆ 4         ┆ 2.394183 ┆ -4.300847   │
└─────────┴─────────┴───────────┴──────────┴─────────────┘
Estimated in-memory size (MB): 8498.19


In [4]:
# feature ideas
# lagged features, rolling features, cross sectional features symbol(x) - avg(rest(x))
# symbol historical behavior
def add_basic_features_polars(frame: pl.DataFrame) -> pl.DataFrame:
    max_time = max(int(frame["time_id"].max()), 1)
    feature_exprs = [pl.col(c) for c in FEATURE_COLS]

    out = frame.with_columns(
        [
            pl.sum_horizontal([pl.col(c).is_null().cast(pl.Int16) for c in FEATURE_COLS])
            .cast(pl.Int16)
            .alias("feature_nan_count"),
            pl.mean_horizontal(feature_exprs).cast(pl.Float32).alias("feature_row_mean"),
            pl.mean_horizontal([pl.col(c).abs() for c in FEATURE_COLS])
            .cast(pl.Float32)
            .alias("feature_row_abs_mean"),
            ((2.0 * np.pi * pl.col("time_id").cast(pl.Float32)) / float(max_time))
            .sin()
            .cast(pl.Float32)
            .alias("time_sin"),
            ((2.0 * np.pi * pl.col("time_id").cast(pl.Float32)) / float(max_time))
            .cos()
            .cast(pl.Float32)
            .alias("time_cos"),
        ]
    )
    return out


def select_top_base_features_by_abs_corr(
    frame: pl.DataFrame,
    candidate_cols: list[str],
    target_col: str,
    top_k: int = 10,
) -> list[str]:
    corr_exprs = [pl.corr(pl.col(c), pl.col(target_col)).abs().alias(c) for c in candidate_cols]
    corr_row = frame.select(corr_exprs).row(0, named=True)

    sorted_feats = sorted(
        candidate_cols,
        key=lambda c: (corr_row[c] if corr_row[c] is not None else -1.0),
        reverse=True,
    )
    return sorted_feats[:top_k]


def add_cross_sectional_features_polars(
    frame: pl.DataFrame,
    selected_base_features: list[str],
) -> pl.DataFrame:
    group_cols = ["date_id", "time_id"]
    cs_exprs = []

    for c in selected_base_features:
        cs_exprs.append(
            (pl.col(c) - pl.col(c).mean().over(group_cols))
            .cast(pl.Float32)
            .alias(f"{c}_cs_demean")
        )
        cs_exprs.append(
            (
                pl.col(c).rank("average").over(group_cols).cast(pl.Float32)
                / pl.len().over(group_cols).cast(pl.Float32)
            )
            .cast(pl.Float32)
            .alias(f"{c}_cs_rank_pct")
        )

    return frame.with_columns(cs_exprs)


def add_prev_day_responder_lags_polars(
    frame: pl.DataFrame,
    responder_cols: list[str],
) -> pl.DataFrame:
    # Emulate lags.parquet availability: previous-date responder values by symbol.
    day_last_exprs = [
        pl.col(c).last().cast(pl.Float32).alias(f"{c}_day_last")
        for c in responder_cols
    ]

    daily = frame.group_by(["symbol_id", "date_id"]).agg(day_last_exprs).sort(["symbol_id", "date_id"])

    lag_cols = [f"lag1d_{c}" for c in responder_cols]
    lag_exprs = [
        pl.col(f"{c}_day_last")
        .shift(1)
        .over("symbol_id")
        .cast(pl.Float32)
        .alias(f"lag1d_{c}")
        for c in responder_cols
    ]

    lag_df = daily.with_columns(lag_exprs).select(["symbol_id", "date_id"] + lag_cols)

    out = frame.join(lag_df, on=["symbol_id", "date_id"], how="left")

    out = out.with_columns(
        [
            pl.mean_horizontal([pl.col(c) for c in lag_cols])
            .cast(pl.Float32)
            .alias("lag1d_resp_mean"),
            pl.mean_horizontal([pl.col(c).abs() for c in lag_cols])
            .cast(pl.Float32)
            .alias("lag1d_resp_abs_mean"),
            (
                pl.col("lag1d_responder_6")
                - pl.mean_horizontal([pl.col(c) for c in lag_cols])
            )
            .cast(pl.Float32)
            .alias("lag1d_target_vs_resp_mean"),
        ]
    )

    return out


def weighted_zero_mean_r2(y_true: np.ndarray, y_pred: np.ndarray, w: np.ndarray) -> float:
    num = np.sum(w * (y_true - y_pred) ** 2)
    den = np.sum(w * (y_true**2))
    if den == 0:
        return np.nan
    return 1.0 - num / den


def make_time_series_folds(
    unique_dates: np.ndarray,
    n_folds: int = 3,
    val_days: int = 180,
    gap: int = 1,
    train_lookback_days: int | None = 540,
):
    """
    Build chronological folds with fixed-size validation blocks near the dataset tail.
    Each fold validation span is `val_days` (about 6 months).
    """
    n_dates = len(unique_dates)
    required = n_folds * val_days + gap + 30
    if n_dates < required:
        raise ValueError(
            f"Not enough dates ({n_dates}) for {n_folds} folds with val_days={val_days}."
        )

    folds = []
    for k in range(n_folds):
        # Older fold first, newest fold last
        val_end = n_dates - (n_folds - 1 - k) * val_days
        val_start = val_end - val_days
        train_end = val_start - gap

        if train_lookback_days is None:
            train_start = 0
        else:
            train_start = max(0, train_end - train_lookback_days)

        train_dates = unique_dates[train_start:train_end]
        val_dates = unique_dates[val_start:val_end]

        if len(train_dates) == 0 or len(val_dates) == 0:
            raise ValueError("Empty train/val split created. Increase MAX_DATES.")

        folds.append((train_dates, val_dates))

    return folds

In [5]:
# Build baseline engineered dataset first.
if "df_pl" not in globals():
    scan = build_scan()
    df_pl = scan.collect(streaming=True).sort(["date_id", "time_id", "symbol_id"])

df_base_pl = add_basic_features_polars(df_pl)
del df_pl
gc.collect()

unique_dates = np.sort(df_base_pl["date_id"].unique().to_numpy())
folds = make_time_series_folds(
    unique_dates,
    n_folds=N_FOLDS,
    val_days=VAL_DAYS,
    gap=DATE_GAP,
    train_lookback_days=TRAIN_LOOKBACK_DAYS,
)

for i, (tr_dates, va_dates) in enumerate(folds, start=1):
    print(
        f"Fold {i}: train [{int(tr_dates.min())}, {int(tr_dates.max())}] ({len(tr_dates)} dates) | "
        f"val [{int(va_dates.min())}, {int(va_dates.max())}] ({len(va_dates)} dates, ~6 months)"
    )

if PHASE1_ENABLE:
    # Use first fold train window to pick a compact set of base features.
    phase1_train_dates = folds[0][0]
    phase1_train_pl = df_base_pl.filter(pl.col("date_id").is_in(phase1_train_dates))
    phase1_top_features = select_top_base_features_by_abs_corr(
        phase1_train_pl,
        FEATURE_COLS,
        TARGET,
        top_k=PHASE1_TOPK_BASE_FEATURES,
    )
    print(f"Phase 1 selected base features ({len(phase1_top_features)}): {phase1_top_features}")

    df_fe_pl = add_cross_sectional_features_polars(df_base_pl, phase1_top_features)
    del phase1_train_pl
else:
    phase1_top_features = []
    df_fe_pl = df_base_pl

if PHASE3_ENABLE:
    if PHASE3_USE_RESPONDER_LAGS:
        df_fe_pl = add_prev_day_responder_lags_polars(df_fe_pl, RESPONDER_COLS)

# Drop intermediate baseline frame once final feature frame is built.
if df_fe_pl is not df_base_pl:
    del df_base_pl
gc.collect()

MODEL_FEATURES = [
    c
    for c in df_fe_pl.columns
    if c not in {TARGET, WEIGHT_COL} and not c.startswith("responder_")
]

print(f"Model features after Phase 1+2+3: {len(MODEL_FEATURES)}")
print(f"Feature-engineered size (MB): {df_fe_pl.estimated_size('mb'):.2f}")

Fold 1: train [978, 1517] (540 dates) | val [1519, 1698] (180 dates, ~6 months)
Phase 1 selected base features (10): ['feature_06', 'feature_36', 'feature_04', 'feature_07', 'feature_01', 'feature_05', 'feature_45', 'feature_56', 'feature_60', 'feature_33']
Model features after Phase 1: 107
Feature-engineered size (MB): 11002.52


In [15]:
df_fe_pl.sample(5)

date_id,time_id,symbol_id,weight,responder_6,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,…,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_nan_count,feature_row_mean,feature_row_abs_mean,time_sin,time_cos,feature_06_cs_demean,feature_06_cs_rank_pct,feature_36_cs_demean,feature_36_cs_rank_pct,feature_04_cs_demean,feature_04_cs_rank_pct,feature_07_cs_demean,feature_07_cs_rank_pct,feature_01_cs_demean,feature_01_cs_rank_pct,feature_05_cs_demean,feature_05_cs_rank_pct,feature_45_cs_demean,feature_45_cs_rank_pct,feature_56_cs_demean,feature_56_cs_rank_pct,feature_60_cs_demean,feature_60_cs_rank_pct,feature_33_cs_demean,feature_33_cs_rank_pct
i16,i16,i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
1106,385,1,2.976141,0.135331,1.071146,-0.144927,1.155914,1.447558,0.549787,0.177113,-0.287571,-0.196063,-0.327381,11.0,7.0,76.0,0.0798,-0.395535,-0.517684,-0.176947,-0.456113,-0.43507,0.986402,-0.5779,0.164258,-0.004605,2.268754,1.478503,-1.136294,-0.287858,0.297712,1.335492,1.212558,-0.330177,-0.080906,-0.004702,…,0.161995,-0.321931,-0.614903,-0.049776,-0.401926,-0.449524,-0.352113,-0.421994,-0.173403,-0.288119,-0.228079,-0.284151,0,1.224957,1.789564,0.597207,-0.802087,0.019006,0.538462,-0.447853,0.25641,0.202456,0.794872,-0.028295,0.307692,0.194409,0.666667,0.036364,0.692308,-2.002419,0.025641,-2.600444,0.025641,0.013898,0.564103,-0.449612,0.205128
1276,118,31,0.922165,0.183739,0.370537,1.203732,-0.142616,-0.009852,-1.611941,-0.608034,0.351663,-0.646265,0.22666,44.0,3.0,24.0,-0.188586,-0.141405,-0.462813,-0.535658,-0.571781,-0.762561,-0.665267,-0.639501,-1.141511,0.056146,-1.189231,-0.78138,0.944823,-0.364081,-0.755278,-2.001576,-2.299823,-0.599358,-0.553379,0.080199,…,-0.138005,-0.197083,-0.388794,-0.159387,-0.189724,-0.577613,-0.178057,-0.202277,-0.286007,-0.370795,-0.177453,-0.359872,0,0.647336,1.485895,0.693775,0.720192,0.019227,0.615385,-0.421917,0.333333,0.043006,0.564103,-0.032227,0.461538,-0.100219,0.410256,-0.13366,0.179487,-0.427716,0.153846,-0.272131,0.358974,-0.209399,0.282051,2.017834,0.923077
1295,434,14,1.491668,-0.269867,-0.012287,0.645163,0.051993,-0.258006,0.745032,-0.299757,-0.439732,-0.733054,0.142606,44.0,3.0,16.0,-0.681919,-0.440557,-0.527625,-0.50552,-0.515648,-0.445714,0.187536,-1.650105,-1.150179,-0.2191,-1.247497,-0.779893,-0.328787,-1.118808,-1.029542,-0.307435,-0.295206,-0.525565,-0.497129,-0.151086,…,-0.738922,-0.28909,-0.513752,-0.621213,-0.455119,-1.037215,-0.167456,-0.215199,0.565017,0.318056,-0.018007,-0.070764,0,0.516878,1.247691,0.316115,-0.948721,-0.088381,0.184211,-0.274655,0.394737,0.092169,0.578947,-0.112993,0.289474,-0.19938,0.236842,-0.050507,0.263158,-0.230136,0.421053,0.130754,0.526316,0.083944,0.5,0.250448,0.684211
1366,153,22,1.247092,-0.495437,1.596982,1.931442,1.385233,1.023893,0.705063,1.381194,0.296479,0.170824,-0.281627,30.0,10.0,13.0,-0.319847,-0.139612,-0.542133,-0.464877,-0.629344,-0.722957,-0.605411,0.34494,0.504533,-0.109246,-0.140738,-0.266499,0.503914,1.426289,0.496845,-0.152117,-0.498653,-0.55754,-0.61193,-0.11412,…,-0.348609,-0.201685,-0.499213,-0.243521,-0.022206,-0.46261,0.062416,0.055787,0.623711,0.627232,0.122474,0.118291,0,0.580235,1.3117,0.838287,0.545229,-0.128956,0.153846,0.63539,0.74359,-0.559875,0.051282,-0.123368,0.025641,0.077055,0.641026,0.077979,0.641026,-1.506965,0.051282,-1.257454,0.128205,-0.623593,0.153846,-1.181846,0.153846
1394,23

In [ ]:
params = {
    "objective": "regression",
    "learning_rate": 0.05,
    "n_estimators": 800,
    "num_leaves": 64,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
}

if IS_KAGGLE:
    params.update(
        {
            "device": "gpu",
            "gpu_use_dp": False,
            "max_bin": 255,
            "force_col_wise": True,
        }
    )
    print("Using LightGBM GPU mode on Kaggle.")
else:
    params.update({"force_col_wise": True})
    print("Using LightGBM CPU mode.")

fold_metrics = []
global_val_num = 0.0
global_val_den = 0.0

for fold_idx, (train_dates, val_dates) in enumerate(folds, start=1):
    train_pl = df_fe_pl.filter(pl.col("date_id").is_in(train_dates))
    val_pl = df_fe_pl.filter(pl.col("date_id").is_in(val_dates))

    medians = train_pl.select([pl.col(c).median().alias(c) for c in MODEL_FEATURES]).row(
        0, named=True
    )
    fill_exprs = [
        pl.col(c)
        .fill_null(medians[c])
        .fill_nan(medians[c])
        .cast(pl.Float32)
        .alias(c)
        for c in MODEL_FEATURES
    ]

    train_pl = train_pl.with_columns(fill_exprs)
    val_pl = val_pl.with_columns(fill_exprs)

    print(
        f"Fold {fold_idx} rows | train={train_pl.height:,}, val={val_pl.height:,} | "
        f"train_mb~{train_pl.estimated_size('mb'):.1f}, val_mb~{val_pl.estimated_size('mb'):.1f}"
    )

    X_train = np.ascontiguousarray(train_pl.select(MODEL_FEATURES).to_numpy(), dtype=np.float32)
    y_train = np.ascontiguousarray(train_pl[TARGET].to_numpy(), dtype=np.float32)
    w_train = np.ascontiguousarray(train_pl[WEIGHT_COL].to_numpy(), dtype=np.float32)

    X_val = np.ascontiguousarray(val_pl.select(MODEL_FEATURES).to_numpy(), dtype=np.float32)
    y_val = np.ascontiguousarray(val_pl[TARGET].to_numpy(), dtype=np.float32)
    w_val = np.ascontiguousarray(val_pl[WEIGHT_COL].to_numpy(), dtype=np.float32)

    # Free Polars fold slices before fit to reduce peak memory.
    del train_pl, val_pl
    gc.collect()

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train,
        y_train,
        sample_weight=w_train,
        eval_set=[(X_val, y_val)],
        eval_sample_weight=[w_val],
        eval_metric="l2",
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )

    train_pred = model.predict(X_train, num_iteration=model.best_iteration_)
    val_pred = model.predict(X_val, num_iteration=model.best_iteration_)

    train_r2 = weighted_zero_mean_r2(y_train, train_pred, w_train)
    val_r2 = weighted_zero_mean_r2(y_val, val_pred, w_val)

    val_num = float(np.sum(w_val * (y_val - val_pred) ** 2))
    val_den = float(np.sum(w_val * (y_val**2)))
    global_val_num += val_num
    global_val_den += val_den

    fold_metrics.append(
        {
            "fold": fold_idx,
            "best_iteration": int(model.best_iteration_ or params["n_estimators"]),
            "train_weighted_zero_mean_r2": train_r2,
            "val_weighted_zero_mean_r2": val_r2,
        }
    )

    print(
        f"Fold {fold_idx} | best_iter={fold_metrics[-1]['best_iteration']} | "
        f"train_r2={train_r2:.6f} | val_r2={val_r2:.6f}"
    )

    del X_train, X_val, y_train, y_val, w_train, w_val, train_pred, val_pred, model
    gc.collect()

metrics_df = pd.DataFrame(fold_metrics)
metrics_df

Fold 1 rows | train=19,639,752, val=6,686,944 | train_mb~8238.8, val_mb~2807.1


In [ ]:
overall_oof_r2 = np.nan if global_val_den == 0 else 1.0 - (global_val_num / global_val_den)

print("\nTrain/Eval metrics summary")
print(metrics_df.to_string(index=False))
print(f"\nMean fold train weighted zero-mean R^2: {metrics_df['train_weighted_zero_mean_r2'].mean():.6f}")
print(f"Mean fold val weighted zero-mean R^2: {metrics_df['val_weighted_zero_mean_r2'].mean():.6f}")
print(f"Overall OOF weighted zero-mean R^2: {overall_oof_r2:.6f}")